# RAM-H1200 Bone Segmentation Experiment
### DenseNet121 -> LayerCAM -> Bone Morphology -> SAM ViT-B -> Quantitative Evaluation / U-Net

This notebook is an executable experiment report. It explains every stage, runs the matching source-code command, and visualizes image-processing outputs so the reader can follow the pipeline from top to bottom.

Dataset: **RAM-H1200 only**. Target: **visible hand bones in X-ray images**, not the full soft-tissue hand silhouette.

## 0. Pipeline Overview

```text
RAM-H1200 hand X-ray
  -> DenseNet121 hand checkpoint
  -> LayerCAM localization
  -> bone-specific morphology
  -> SAM box/point prompting
  -> bone-aware mask selection
  -> pseudo bone mask
  -> Dice/IoU against RAM-H1200 ground truth
```

RAM-H1200 provides ground-truth bone masks, so the pseudo-mask branch can be evaluated quantitatively instead of only visually.

## 1. Paths and Run Switches

This cell defines dataset paths, output folders, image size, and which expensive stages should run.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = Path('D:/thesis')
PROJECT_DIR = ROOT / 'project'
RAM_ROOT = ROOT / 'RAM-H1200-v1'
SAM_CHECKPOINT = ROOT / 'sam_vit_b_01ec64.pth'

OUTPUT_ROOT = PROJECT_DIR / 'outputs'
CLASSIFIER_OUTPUT = OUTPUT_ROOT / 'classifier'
PSEUDO_OUTPUT = OUTPUT_ROOT / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_ROOT / 'segmentation'
EVAL_CSV = OUTPUT_ROOT / 'ramh1200_eval.csv'
VIZ_OUTPUT = OUTPUT_ROOT / 'viz'
DEBUG_OUTPUT = OUTPUT_ROOT / 'debug_viz'
ABLATION_OUTPUT = OUTPUT_ROOT / 'ablation_preview'
MORPH_DEBUG_OUTPUT = OUTPUT_ROOT / 'morphology_debug'

IMAGE_SIZE = 384
NUM_WORKERS = 2
BATCH_SIZE_CLASSIFIER = 4
BATCH_SIZE_SEGMENTATION = 4
EPOCHS_CLASSIFIER = 25
EPOCHS_SEGMENTATION = 25

# Ch?y l?n ??u: ?? True ?? c? checkpoint classifier.
# Khi ?? c? checkpoint/log, c? th? ??i False ?? ti?t ki?m th?i gian.
RUN_INSTALL = False
RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PSEUDO_MASKS = False   # False = preview 10 ?nh, True = ch?y full validation split
RUN_EVALUATE_PSEUDO = True
RUN_TRAIN_UNET = False
RUN_VISUALIZE_SAMPLE = True
RUN_DEBUG_SAM = True
RUN_ABLATION_PREVIEW = False
RUN_MORPHOLOGY_EXPLAINER = True

print('PROJECT_DIR:', PROJECT_DIR)
print('RAM_ROOT:', RAM_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Local Environment Setup

This stage verifies that the notebook is running inside the project source tree and that the main scripts are available.

In [ ]:
os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())

if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
else:
    print('RUN_INSTALL=False, skipping c?i ??t package.')

print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Kh?ng import ???c torch:', repr(exc))

required_files = [
    PROJECT_DIR / 'datasets' / 'ramh1200.py',
    PROJECT_DIR / 'train_classifier.py',
    PROJECT_DIR / 'generate_pseudo_masks.py',
    PROJECT_DIR / 'evaluate_ramh1200_masks.py',
    PROJECT_DIR / 'train_segmentation.py',
    PROJECT_DIR / 'visualize_pipeline.py',
]
for path in required_files:
    print(('OK  ' if path.exists() else 'MISS'), path)

## 3. Dataset Inspection: RAM-H1200

**Purpose:** verify that the notebook reads RAM-H1200 correctly and that the annotations are COCO RLE bone masks.

**Input:** `RAM_ROOT/Segmentation/<split>`.

**Expected visual output:** one original X-ray, its ground-truth bone mask, and a mask overlay.

In [ ]:
import json
from pathlib import Path

if not RAM_ROOT.exists():
    raise FileNotFoundError(f'Kh?ng t?m th?y RAM_ROOT: {RAM_ROOT}')

SEG_ROOT = RAM_ROOT / 'Segmentation' if (RAM_ROOT / 'Segmentation').exists() else RAM_ROOT
if not (SEG_ROOT / 'train').exists():
    raise FileNotFoundError(f'Kh?ng th?y split train trong {SEG_ROOT}')

print('SEG_ROOT:', SEG_ROOT)
for split in ['train', 'val', 'validation', 'test']:
    split_dir = SEG_ROOT / split
    if split_dir.exists():
        ann_path = split_dir / '_annotations_bone_rle.coco.json'
        images = sorted([p for p in split_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}])
        print(f'{split:<10} images={len(images):>5} annotation={ann_path.exists()}')
        if ann_path.exists():
            data = json.loads(ann_path.read_text(encoding='utf-8'))
            cats = [c.get('name', c.get('id')) for c in data.get('categories', [])]
            print('  categories:', cats[:30])
            print('  coco images:', len(data.get('images', [])), 'annotations:', len(data.get('annotations', [])))

In [ ]:
from datasets.ramh1200 import RAMH1200SegmentationDataset
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

preview_ds = RAMH1200SegmentationDataset(root=RAM_ROOT, split='val', image_size=IMAGE_SIZE)
image_tensor, mask_tensor, image_name = preview_ds[0]
sample = preview_ds.samples[0]
image_path = Path(sample['image_path'])

original = Image.open(image_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
gt_mask = mask_tensor[0].numpy() > 0.5
original_np = np.array(original)
overlay = original_np.copy()
overlay[gt_mask] = (0.55 * overlay[gt_mask] + 0.45 * np.array([255, 40, 40])).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(original_np)
axes[0].set_title(f'Original\n{image_name}')
axes[1].imshow(gt_mask, cmap='gray')
axes[1].set_title('GT bone mask')
axes[2].imshow(overlay)
axes[2].set_title('Overlay GT l?n ?nh')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('image tensor:', tuple(image_tensor.shape), image_tensor.dtype)
print('mask tensor:', tuple(mask_tensor.shape), mask_tensor.dtype, 'mask pixels:', int(mask_tensor.sum().item()))

## 4. SAM Checkpoint

**Purpose:** Stage 2 needs SAM ViT-B to refine bone regions from box and point prompts.

**Input:** `sam_vit_b_01ec64.pth`.

**Output:** a valid checkpoint path before pseudo-mask generation starts.

In [ ]:
if not SAM_CHECKPOINT.exists():
    print('Ch?a th?y SAM checkpoint, ?ang t?i SAM ViT-B...')
    import urllib.request
    SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    urllib.request.urlretrieve(url, SAM_CHECKPOINT)
else:
    print('SAM checkpoint ?? t?n t?i:', SAM_CHECKPOINT)

## 5. Stage 1: DenseNet121 Hand Checkpoint

**Pipeline role:** provide feature maps and gradients for LayerCAM.

**Input:** RAM-H1200 hand X-ray images.

**Training target:** a single image-level label, `hand = 1`.

**Output:** `best_classifier.pt` and `training_log.csv`.

In [ ]:
classifier_cmd = [
    sys.executable, 'train_classifier.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_CLASSIFIER),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_CLASSIFIER),
    '--output-dir', str(CLASSIFIER_OUTPUT),
]
print(' '.join(classifier_cmd))
if RUN_TRAIN_CLASSIFIER:
    subprocess.run(classifier_cmd, check=True)
else:
    print('RUN_TRAIN_CLASSIFIER=False, skipping train classifier.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / 'best_classifier.pt'
log_path = CLASSIFIER_OUTPUT / 'training_log.csv'
print('Classifier checkpoint:', CLASSIFIER_CHECKPOINT, 'exists=', CLASSIFIER_CHECKPOINT.exists())
if log_path.exists():
    df = pd.read_csv(log_path)
    display(df.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[['train_loss', 'val_loss']].plot(ax=axes[0], title='Classifier loss')
    df[['train_f1', 'val_f1']].plot(ax=axes[1], title='Classifier F1')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ch?a c? training_log.csv cho classifier.')

## 6. Stage 2: Pseudo-Mask Generation with LayerCAM + Bone Morphology + SAM

**Pipeline role:** generate a bone pseudo mask from an X-ray image.

**Internal processing:** DenseNet features -> LayerCAM -> bone morphology -> SAM prompts -> SAM candidates -> bone-aware selection -> conservative refinement.

**Output:** binary pseudo masks in `pseudo_masks/masks` and LayerCAM overlays in `pseudo_masks/overlays`.

In [ ]:
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(f'Thi?u classifier checkpoint: {CLASSIFIER_CHECKPOINT}. H?y ch?y Stage 1 tr??c.')

process_args = ['--process-all'] if RUN_FULL_PSEUDO_MASKS else ['--max-images', '10']
pseudo_cmd = [
    sys.executable, 'generate_pseudo_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', '1',
    '--num-workers', str(NUM_WORKERS),
    '--confidence-threshold', '0.5',
    '--cam-percentile', '85.0',
    '--max-points', '5',
    '--min-component-area', '100',
    '--mask-score-threshold', '0.4',
    '--selection-method', 'bone_hybrid',
    '--fusion-topk', '3',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--max-bone-components', '12',
    '--points-per-component', '3',
    '--bbox-padding-ratio', '0.02',
    '--negative-points-per-component', '0',
    '--bone-seed-percentile', '88',
    '--bone-support-percentile', '68',
    '--closing-kernel', '5',
    '--opening-kernel', '0',
    '--max-hole-area', '500',
    '--min-size', '40',
    '--save-visuals-limit', '10',
    *process_args,
    '--output-dir', str(PSEUDO_OUTPUT),
]
print(' '.join(pseudo_cmd))
subprocess.run(pseudo_cmd, check=True)

### 6.1 Visual Check: Original Image, LayerCAM Overlay, and Pseudo Mask

Inspect this before running a full split. The CAM should activate on bone regions, and the pseudo mask should cover visible bones without becoming a full soft-tissue hand silhouette.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

mask_dir = PSEUDO_OUTPUT / 'masks'
overlay_dir = PSEUDO_OUTPUT / 'overlays'
mask_files = sorted(mask_dir.glob('*.png'))
print('Number of pseudo masks:', len(mask_files), mask_dir)

n = min(5, len(mask_files))
if n:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1:
        axes = np.array([axes])
    val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
    for row, mask_path in zip(axes, mask_files[:n]):
        stem = mask_path.stem
        image_match = next(iter(sorted(val_dir.glob(f'{stem}.*'))), None)
        overlay_match = next(iter(sorted(overlay_dir.glob(f'{stem}*fused_layercam.png'))), None)
        if image_match:
            row[0].imshow(Image.open(image_match).convert('RGB'))
            row[0].set_title(f'Original\n{stem}')
        if overlay_match:
            row[1].imshow(Image.open(overlay_match).convert('RGB'))
            row[1].set_title('LayerCAM overlay')
        row[2].imshow(Image.open(mask_path), cmap='gray')
        row[2].set_title('Pseudo mask')
        for ax in row:
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No pseudo masks were found.')

### 6.2 Image-Processing Explainer: From X-ray to Bone-Guided Prompt Map

This cell visualizes the image-processing part before SAM: original X-ray, LayerCAM, CAM+bone guidance, enhanced grayscale, cortical edges, bone likelihood, seeds, reconstructed support, and prompt points.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = Path('D:/thesis')
PROJECT_DIR = ROOT / 'project'
RAM_ROOT = ROOT / 'RAM-H1200-v1'
SAM_CHECKPOINT = ROOT / 'sam_vit_b_01ec64.pth'

OUTPUT_ROOT = PROJECT_DIR / 'outputs'
CLASSIFIER_OUTPUT = OUTPUT_ROOT / 'classifier'
PSEUDO_OUTPUT = OUTPUT_ROOT / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_ROOT / 'segmentation'
EVAL_CSV = OUTPUT_ROOT / 'ramh1200_eval.csv'
VIZ_OUTPUT = OUTPUT_ROOT / 'viz'
DEBUG_OUTPUT = OUTPUT_ROOT / 'debug_viz'
ABLATION_OUTPUT = OUTPUT_ROOT / 'ablation_preview'
MORPH_DEBUG_OUTPUT = OUTPUT_ROOT / 'morphology_debug'

IMAGE_SIZE = 384
NUM_WORKERS = 2
BATCH_SIZE_CLASSIFIER = 4
BATCH_SIZE_SEGMENTATION = 4
EPOCHS_CLASSIFIER = 25
EPOCHS_SEGMENTATION = 25

# Ch?y l?n ??u: ?? True ?? c? checkpoint classifier.
# Khi ?? c? checkpoint/log, c? th? ??i False ?? ti?t ki?m th?i gian.
RUN_INSTALL = False
RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PSEUDO_MASKS = False   # False = preview 10 ?nh, True = ch?y full validation split
RUN_EVALUATE_PSEUDO = True
RUN_TRAIN_UNET = False
RUN_VISUALIZE_SAMPLE = True
RUN_DEBUG_SAM = True
RUN_ABLATION_PREVIEW = False
RUN_MORPHOLOGY_EXPLAINER = True

print('PROJECT_DIR:', PROJECT_DIR)
print('RAM_ROOT:', RAM_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 7. Quantitative Evaluation: Pseudo Mask vs RAM-H1200 Ground Truth

**Purpose:** measure pseudo-mask quality with Dice and IoU.

If `RUN_FULL_PSEUDO_MASKS=False`, the result is only a 10-image preview metric. For thesis/report numbers, set `RUN_FULL_PSEUDO_MASKS=True`, rerun Stage 2, then rerun this stage.

In [ ]:
eval_cmd = [
    sys.executable, 'evaluate_ramh1200_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--pred-mask-root', str(PSEUDO_OUTPUT / 'masks'),
    '--image-size', str(IMAGE_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--output-csv', str(EVAL_CSV),
]
print(' '.join(eval_cmd))
if RUN_EVALUATE_PSEUDO:
    subprocess.run(eval_cmd, check=True)
else:
    print('RUN_EVALUATE_PSEUDO=False, skipping evaluate.')

In [ ]:
if EVAL_CSV.exists():
    eval_df = pd.read_csv(EVAL_CSV)
    display(eval_df.tail(10))
    valid = eval_df[eval_df['status'] == 'ok'].copy()
    if not valid.empty:
        print('S? ?nh evaluate:', len(valid))
        print('Mean Dice:', valid['dice'].mean())
        print('Mean IoU :', valid['iou'].mean())
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        valid['dice'].hist(ax=axes[0], bins=20)
        axes[0].set_title('Ph?n b? Dice pseudo mask')
        valid['iou'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Ph?n b? IoU pseudo mask')
        plt.tight_layout()
        plt.show()
else:
    print('Evaluation CSV was not found.')

## 8. Stage 3: Supervised U-Net Baseline

**Pipeline role:** train a fully supervised U-Net using RAM-H1200 ground-truth masks.

**Input:** X-ray image and GT bone mask.

**Output:** `best_unet.pt` and loss/Dice/IoU curves.

In [ ]:
unet_cmd = [
    sys.executable, 'train_segmentation.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_SEGMENTATION),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_SEGMENTATION),
    '--output-dir', str(SEG_OUTPUT),
]
print(' '.join(unet_cmd))
if RUN_TRAIN_UNET:
    subprocess.run(unet_cmd, check=True)
else:
    print('RUN_TRAIN_UNET=False, skipping train U-Net.')

In [ ]:
seg_log = SEG_OUTPUT / 'training_log.csv'
seg_ckpt = SEG_OUTPUT / 'best_unet.pt'
print('U-Net checkpoint:', seg_ckpt, 'exists=', seg_ckpt.exists())
if seg_log.exists():
    seg_df = pd.read_csv(seg_log)
    display(seg_df.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    seg_df[['train_loss', 'val_loss']].plot(ax=axes[0], title='U-Net loss')
    seg_df[['train_dice', 'val_dice']].plot(ax=axes[1], title='U-Net Dice')
    seg_df[['train_iou', 'val_iou']].plot(ax=axes[2], title='U-Net IoU')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('U-Net training_log.csv was not found.')

## 9. Full Pipeline Visualization on One Image

This stage creates a pipeline strip with `visualize_pipeline.py`: original image, LayerCAM, bone guidance, prompts, SAM candidates, pseudo mask, and optionally U-Net output.

In [ ]:
val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
sample_image = next(iter(sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg'}])), None)
print('Sample image:', sample_image)

viz_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(sample_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--output-path', str(VIZ_OUTPUT / f'{sample_image.stem}_pipeline.png'),
]
if (SEG_OUTPUT / 'best_unet.pt').exists():
    viz_cmd.extend(['--segmentation-checkpoint', str(SEG_OUTPUT / 'best_unet.pt')])
print(' '.join(viz_cmd))
if RUN_VISUALIZE_SAMPLE and sample_image is not None:
    subprocess.run(viz_cmd, check=True)
else:
    print('RUN_VISUALIZE_SAMPLE=False ho?c kh?ng t?m th?y ?nh m?u.')

In [ ]:
viz_files = sorted(VIZ_OUTPUT.glob('*_pipeline.png'))
if viz_files:
    img = Image.open(viz_files[-1])
    plt.figure(figsize=(22, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(viz_files[-1].name)
    plt.show()
else:
    print('No pipeline strip was found.')

## 10. SAM Candidate Debugging

This is the failure-analysis stage. It saves and displays every SAM candidate mask, each overlay, and `scores.json`. Use it when the pseudo mask is incomplete, too large, or follows soft tissue instead of bone.

In [ ]:
debug_image = sample_image
DEBUG_OUTPUT.mkdir(parents=True, exist_ok=True)
debug_strip = DEBUG_OUTPUT / f'{debug_image.stem}_debug_pipeline.png'

debug_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(debug_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--debug',
    '--output-path', str(debug_strip),
]
print(' '.join(debug_cmd))
if RUN_DEBUG_SAM:
    subprocess.run(debug_cmd, check=True)
else:
    print('RUN_DEBUG_SAM=False, skipping debug SAM.')

In [ ]:
import json

if debug_strip.exists():
    plt.figure(figsize=(22, 4))
    plt.imshow(Image.open(debug_strip))
    plt.axis('off')
    plt.title('Debug pipeline strip')
    plt.show()

debug_dir = DEBUG_OUTPUT / 'debug' / debug_image.stem
mask_files = sorted(debug_dir.glob('mask_*.png'))
overlay_files = sorted(debug_dir.glob('overlay_mask_*.png'))
scores_path = debug_dir / 'scores.json'
print('debug_dir:', debug_dir)
print('SAM candidate masks:', len(mask_files))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])
    for i, mf in enumerate(mask_files):
        axes[0][i].imshow(Image.open(mf), cmap='gray')
        axes[0][i].set_title(mf.stem)
        axes[0][i].axis('off')
        if i < len(overlay_files):
            axes[1][i].imshow(Image.open(overlay_files[i]).convert('RGB'))
        axes[1][i].set_title('overlay')
        axes[1][i].axis('off')
    plt.tight_layout()
    plt.show()

if scores_path.exists():
    scores = json.loads(scores_path.read_text(encoding='utf-8'))
    print('scores.json')
    for key, value in scores.items():
        score = value.get('score', 0)
        area = value.get('area', 0)
        bar = '#' * int(score * 30)
        print(f'{key:<8} score={score:.3f} area={area:>8} {bar}')

## 11. Small Ablation Preview

Optional. Enable `RUN_ABLATION_PREVIEW=True` to compare a few prompt/scoring strategies on the same image before committing to a full run.

In [ ]:
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)
ABLATION_STRATEGIES = [
    ('box_point_bone_hybrid', ['--sam-prompt-mode', 'box_point', '--selection-method', 'bone_hybrid']),
    ('point_bone_hybrid', ['--sam-prompt-mode', 'point', '--selection-method', 'bone_hybrid']),
    ('box_point_coverage', ['--sam-prompt-mode', 'box_point', '--selection-method', 'coverage']),
]

if RUN_ABLATION_PREVIEW:
    for label, extra in ABLATION_STRATEGIES:
        out_path = ABLATION_OUTPUT / f'{label}_pipeline.png'
        cmd = [
            sys.executable, 'visualize_pipeline.py',
            '--image-path', str(debug_image),
            '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
            '--sam-checkpoint', str(SAM_CHECKPOINT),
            '--image-size', str(IMAGE_SIZE),
            '--morphology-fusion-mode', 'components',
            '--output-path', str(out_path),
            *extra,
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)
else:
    print('RUN_ABLATION_PREVIEW=False, skipping ablation preview.')

In [ ]:
ablation_files = sorted(ABLATION_OUTPUT.glob('*_pipeline.png'))
if ablation_files:
    fig, axes = plt.subplots(len(ablation_files), 1, figsize=(22, 4*len(ablation_files)))
    if len(ablation_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, ablation_files):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No ablation output was found.')

## 12. Reporting Checklist

Save these artifacts for the thesis/report:

1. Dataset sample: original X-ray + GT bone mask + overlay.
2. Stage 2 sample: original + LayerCAM overlay + pseudo mask.
3. Morphology explainer: grayscale, edge, likelihood, seeds, support, prompt points.
4. Pipeline strip from `visualize_pipeline.py`.
5. Dice/IoU table from `evaluate_ramh1200_masks.py`.
6. Supervised U-Net training curves and best validation Dice/IoU.
7. Optional ablation figures for prompt mode or scoring method.